# 02 - Exploratory data analysis

Inputs: `train` and `student_test`. Label-based analysis (alignment of GPS to the reference route) uses `train` only.
Regenerate the tables and figures with `python scripts/run_eda.py`; the findings are written up in `reports/data_quality_log.md`.

In [ ]:
import os, sys
from pathlib import Path
# run from the repository root so relative paths in configs work
if Path.cwd().name == "notebooks":
    os.chdir("..")
import numpy as np
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_columns", 60)

## Raw files of one trip

In [ ]:
from pathsnap.data.loader import load_trip
from pathsnap.data.validation import validate_trip
trip = load_trip("train", "00000001")
display(trip.track.head(), trip.nodes.head(), trip.arcs.head(), trip.route.head())
validate_trip(trip)

## Data quality per trip

In [ ]:
q = pd.read_csv("results/data_quality.csv", dtype={"trip_id": str})
q.groupby("partition")[["n_obs", "duration_s", "dt_max_s", "share_dt_eq_1s", "n_gaps_gt_30s",
                        "n_nodes", "n_arcs", "n_weak_components", "largest_weak_component_share",
                        "share_arcs_two_way", "route_len", "route_n_breaks", "share_stationary"]].describe().T

In [ ]:
q.sort_values("dt_max_s", ascending=False)[["partition", "trip_id", "n_obs", "dt_max_s", "n_gaps_gt_30s"]].head(8)

## Figures

In [ ]:
for f in ["eda_obs_per_trip", "eda_sampling_intervals", "eda_trip_locations", "eda_trajectory_grid",
          "eda_network_size_connectivity", "eda_train_alignment_distance", "eda_train_candidate_sweep"]:
    print(f); display(Image(f"results/figures/{f}.png"))

## Projection check (local equirectangular vs haversine)

In [ ]:
pd.read_csv("results/projection_check.csv").describe()

## Fixes far from the reference route (train)

In [ ]:
q[q.partition == "train"].sort_values("share_label_excluded", ascending=False)[
    ["trip_id", "n_obs", "share_label_excluded", "share_stationary", "align_dist_p50_m", "align_dist_p95_m"]].head(8)

## Candidate generation

In [ ]:
sweep = pd.read_csv("results/candidate_sweep.csv")
sweep.pivot(index="radius_m", columns="top_k", values="candidate_miss_rate").round(4)

In [ ]:
pd.read_csv("results/candidate_miss_rate_summary.csv", index_col=0).T

## Folium map

Interactive maps are in `maps/html/`. This renders one inline.

In [ ]:
import folium
from pathsnap.graph.road_graph import RoadGraph
from pathsnap.visualisation.folium_maps import trip_map
t = load_trip("train", "00000095")
g = RoadGraph.from_trip(t)
trip_map(g, t.track[["lon", "lat"]].to_numpy(), reference_arcs=t.route["arc_id"].to_numpy(), title="train 00000095")